# Bronze Layer — Data Ingestion Pipeline

This notebook ingests raw source data from CSV files stored in Unity Catalog volumes into the `sales_datalakehouse.bronze` schema. It covers two source systems:

* **Step 1 — CRM Source**: Manually loads customer, product, and sales detail files from the `source_crm` volume folder.
* **Step 2 — ERP Source**: Uses a config-driven loop to load all CSV files from the `source_erp` volume folder.


## Step 1.1 — Load Customer Info from CRM

Read the `cust_info.csv` file from the CRM source volume into a Spark DataFrame with header detection and schema inference enabled.

In [0]:
# Read the customer info CSV from the volume into a Spark DataFrame
file_path = "/Volumes/sales_datalakehouse/bronze/source_system/source_crm/cust_info.csv"

df_cust_info = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

display(df_cust_info)

## Write Customer Data to Bronze Table

Persist the customer DataFrame as a managed Delta table in the bronze schema for downstream processing.

In [0]:
table_name = "sales_datalakehouse.bronze.cust_info"

df_cust_info.write.mode("overwrite").saveAsTable(table_name)

print(f"Saved DataFrame to table: {table_name}")

## Verify Customer Data

Run a quick `SELECT *` to verify the customer data was written correctly to the bronze table.

In [0]:
%sql
select * from sales_datalakehouse.bronze.cust_info


## Step 1.2 — Load Product Info from CRM

Read the `prd_info.csv` file from the CRM source volume and write it as a managed Delta table in the bronze schema.

In [0]:
# Read the product info CSV from the volume into a Spark DataFrame
file_path = "/Volumes/sales_datalakehouse/bronze/source_system/source_crm/prd_info.csv"

df_prd_info = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

display(df_prd_info)

In [0]:
table_name = "sales_datalakehouse.bronze.bronze_crm_prd_info"

df_prd_info.write.mode("overwrite").saveAsTable(table_name)

print(f"Saved DataFrame to table: {table_name}")

## Rename Customer Table

Rename the `cust_info` table to `bronze_crm_cust_info` to follow the bronze layer naming convention (`bronze_<source>_<entity>`).

In [0]:
%sql
ALTER TABLE sales_datalakehouse.bronze.cust_info RENAME TO sales_datalakehouse.bronze.bronze_crm_cust_info;

## Step 1.3 — Load Sales Details from CRM

Read the `sales_details.csv` file from the CRM source volume and write it as a managed Delta table in the bronze schema.

In [0]:
# Read the product info CSV from the volume into a Spark DataFrame
file_path = "/Volumes/sales_datalakehouse/bronze/source_system/source_crm/sales_details.csv"

df_sales_details = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

display(df_sales_details)

In [0]:
table_name = "sales_datalakehouse.bronze.bronze_sales_details"

df_prd_info.write.mode("overwrite").saveAsTable(table_name)

print(f"Saved DataFrame to table: {table_name}")

## Step 2 — Load ERP Source Data Using Ingestion Config

Instead of loading files one by one, this step uses a configuration file (`INGESTION_CONFIG`) that defines all ERP source CSV files and their target table names. A loop iterates over the config entries, reads each CSV from the `source_erp` volume folder, and writes it as a managed Delta table in the `sales_datalakehouse.bronze` schema.

In [0]:
import importlib.util

# Load the ingestion config from the config file
config_path = "/Workspace/Users/prafulk5748@gmail.com/SalesDatalakeHouse/config folder/New File 2026-09-26 23:30:04.py"
spec = importlib.util.spec_from_file_location("ingestion_config", config_path)
config_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config_module)

config = config_module.INGESTION_CONFIG
base_path = config["base_path"]
target_catalog = config["target_catalog"]
target_schema = config["target_schema"]

# Loop through each source file and load it into the bronze schema
for source in config["sources"]:
    file_path = base_path + source["file_name"]
    table_name = f"{target_catalog}.{target_schema}.{source['table_name']}"

    print(f"Reading: {file_path}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    df.write.mode("overwrite").saveAsTable(table_name)

    print(f"Saved to table: {table_name} ({df.count()} rows)")

print("\nERP ingestion complete!")

In [0]:
%sql
ALTER TABLE sales_datalakehouse.bronze.bronze_sales_details RENAME TO sales_datalakehouse.bronze.bronze_crm_sales_details;